# Imports

In [ ]:
import argparse
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

import os
import sys
import csv
import time
import random
import heapq, re, json
from typing import List, Tuple, Set, Iterable, Optional

from letterboxdpy import user as lb_user
from letterboxdpy.utils.utils_file import (
    build_path,
    check_and_create_dirs,
    save_json,
    build_click_url,
)

import json
import os
import pickle
import logging
from pathlib import Path
from typing import Dict, List, Optional

import networkx as nx

import argparse
import csv
import logging
import re
import time
from pathlib import Path

from letterboxdpy.movie import Movie

# Part 0: Data Acquisition
For our project's dataset we have chosen the popular movie focused social media site Letterboxd.
- **Remark:** These codes scrape the data with a different kind of data structure.

## Generating Seed
For our project we have decided on the following:
- We will start with top 10 users of Letterboxd. (All-time popularity: https://letterboxd.com/members/popular/)
- We will increase the network by adding the outgoing-edges' nodes (following) to a priority-queue and always adding the next node with the most followers.
### Some Starting Parameters

In [ ]:
BASE = "https://letterboxd.com/"
HEADERS = {
    # Be a good citizen: identify yourself (put your own contact if you like)
    "User-Agent": "letterboxd-research/1.0 (+support@example.com)"
}

# Known routes for the Popular Members views
SORT_PATHS = {
    "all-time": "members/popular/",
    "week":     "members/popular/this/week/",
    "month":    "members/popular/this/month/",
    "year":     "members/popular/this/year/",
}

### Some Helper Functions

In [ ]:
# A random sleep to be polite with scraping on Letterboxd
def sleep_polite(low=0.8, high=1.6):
    time.sleep(random.uniform(low, high))

# Get url
def get(url, max_retries=6, timeout=20):
    """GET with retry/backoff on 429/5xx."""
    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout)
        except requests.RequestException:
            # Network hiccup → small backoff
            time.sleep(min(8, 2**attempt) + random.random())
            continue

        if r.status_code in (429, 502, 503, 504):
            # Server stressed or rate-limiting → exponential backoff
            time.sleep(min(16, 2**attempt) + random.random())
            continue

        r.raise_for_status()
        return r
    # Final try raises if still failing
    r.raise_for_status()

# Check whether a link looks like a profile
def looks_like_profile_href(href: str) -> bool:
    """
    Accept only '/username/' (exactly two slashes, trailing slash).
    Filters out '/film/...', '/members/...', etc.
    """
    return (
        isinstance(href, str)
        and href.startswith("/")
        and href.endswith("/")
        and href.count("/") == 2
        and not href.startswith("/film/")
        and not href.startswith("/list/")
        and not href.startswith("/journal/")
        and not href.startswith("/members/")
        and not href.startswith("/crew/")
        and not href.startswith("/news/")
    )

def extract_usernames(soup: BeautifulSoup):
    """
    Grab profile links from member cards. Prefer h3 → a,
    fall back to any <a> that matches '/username/'.
    """
    usernames = []

    # Primary: member cards often have <h3><a href="/user/">
    for a in soup.select("h3 a[href]"):
        href = a.get("href", "")
        if looks_like_profile_href(href):
            usernames.append(href.strip("/"))

    # Fallback: scan all anchors (keeps order; dedup later)
    if not usernames:
        for a in soup.select("a[href]"):
            href = a.get("href", "")
            if looks_like_profile_href(href):
                usernames.append(href.strip("/"))

    # Keep order, drop dups
    seen, ordered = set(), []
    for u in usernames:
        if u not in seen:
            seen.add(u)
            ordered.append(u)
    return ordered

def iterate_popular_usernames(period="all-time", max_users=50, max_pages=20):
    """
    Yield usernames from the Popular Members pages until max_users collected.
    """
    path = SORT_PATHS.get(period, SORT_PATHS["all-time"])
    base = urljoin(BASE, path)

    page = 1
    collected = []
    while len(collected) < max_users and page <= max_pages:
        url = base if page == 1 else urljoin(base, f"page/{page}/")
        resp = get(url)
        soup = BeautifulSoup(resp.text, "html.parser")

        batch = extract_usernames(soup)
        # Stop if the page structure changed or we exhausted
        if not batch:
            break

        # Append new names only
        for u in batch:
            if u not in collected:
                collected.append(u)
                if len(collected) >= max_users:
                    break

        page += 1
        sleep_polite()

    return collected[:max_users]

# Output the seed.csv file
def write_seed_csv(usernames, out_path="seed.csv"):
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["username", "profile_url"])
        for u in usernames:
            w.writerow([u, urljoin(BASE, f"{u}/")])

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--period", choices=["all-time", "week", "month", "year"],
                    default="all-time", help="Popularity period to scrape")
    ap.add_argument("--max", type=int, default=10,
                    help="How many usernames to collect")
    ap.add_argument("--out", type=str, default="seed.csv",
                    help="Output CSV path")
    args = ap.parse_args()

    users = iterate_popular_usernames(period=args.period, max_users=args.max)
    write_seed_csv(users, args.out)
    print(f"Wrote {len(users)} usernames to {args.out}")

if __name__ == "__main__":
    main()

## Scraping the Users
**Remark:** This part is very time-consuming, for us this scraping part took one week of running.

In [ ]:
# -------- Parameters --------
MAX_NEW_PER_USER = 200
LOOKUP_FOLLOWER_COUNT_FOR_NEW = True
PROCESSED_LIMIT = 10000

# -------- HELPERS --------
def sleep_polite(low=1, high=2):
    time.sleep(random.uniform(low, high))

def _maybe_int(x) -> Optional[int]:
    try:
        if isinstance(x, int):
            return x
        if isinstance(x, str) and x.isdigit():
            return int(x)
    except Exception:
        pass
    return None

# ---- Priority Queue HELPERS ---------
Number = int
PQItem = Tuple[int, str, str]

# Convert a text number to an int
def _to_int(s: str) -> Number:
    s = (s or "").strip()
    if not s:
        return 0
    s = re.sub(r"[^\d]", "", s)
    return int(s) if s.isdigit() else 0

# Read the top user of the priority-queue
def read_seed_pq(path: str = "seed.csv") -> List[PQItem]:
    pq: List[PQItem] = []
    if not os.path.exists(path):
        return pq
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            u = (row.get("username") or "").strip().strip("/")
            if not u:
                continue
            url = (row.get("profile_url") or f"https://letterboxd.com/{u}/").strip()
            raw_following = row.get("following") or row.get("following_count") or "0"
            try:
                prio = int(str(raw_following).replace(",", ""))
            except ValueError:
                try:
                    prio = int(str(row.get("followers_count", "0")).replace(",", ""))
                except Exception:
                    prio = 0
            heapq.heappush(pq, (-prio, u, url))
    return pq

# Priority-queue Pop
def pq_pop(pq: List[PQItem]) -> Tuple[str, int, str]:
    neg_f, u, p = heapq.heappop(pq)
    return u, -neg_f, p

# Priority-queue Peek
def pq_peek(pq: List[PQItem]) -> Tuple[str, int, str]:
    neg_f, u, p = pq[0]
    return u, -neg_f, p

# Get the follower count of a user
def extract_followers_count(u: lb_user.User) -> Optional[int]:
    for attr in ("followers_count", "follower_count"):
        if hasattr(u, attr):
            v = _maybe_int(getattr(u, attr))
            if v is not None and v >= 0:
                return v
    try:
        data = u.jsonify()
        if isinstance(data, dict):
            for k in ("followers_count", "followerCount", "followers"):
                v = _maybe_int(data.get(k))
                if v is not None:
                    return v
            for blk in ("counts", "stats", "statistics", "profile", "meta"):
                sub = data.get(blk)
                if isinstance(sub, dict):
                    for k in ("followers", "followers_count", "followerCount"):
                        v = _maybe_int(sub.get(k))
                        if v is not None:
                            return v
    except Exception:
        pass
    try:
        lst = lb_user.User.get_followers(u)
        return len(lst) if lst is not None else 0
    except Exception:
        return None

# --- Resume helpers ------------------------------------------------------

def _load_json_if_exists(path_no_ext: str):
    fp = f"{path_no_ext}.json"
    if not os.path.exists(fp):
        return None
    with open(fp, "r", encoding="utf-8") as f:
        return json.load(f)

def _save_checkpoint(state_dir: str, pq: List[PQItem], exported: Set[str], enqueued: Set[str]):
    # Convert to JSON-serializable forms
    pq_list = [[neg, u, url] for (neg, u, url) in pq]
    save_json(build_path(state_dir, "pq"), pq_list)
    save_json(build_path(state_dir, "exported"), sorted(list(exported)))
    save_json(build_path(state_dir, "enqueued"), sorted(list(enqueued)))

def _load_checkpoint(state_dir: str):
    pq_data = _load_json_if_exists(build_path(state_dir, "pq"))
    exp_data = _load_json_if_exists(build_path(state_dir, "exported"))
    enq_data = _load_json_if_exists(build_path(state_dir, "enqueued"))
    if pq_data is None or exp_data is None or enq_data is None:
        return None
    pq: List[PQItem] = [(int(neg), str(u), str(url)) for neg, u, url in pq_data]
    heapq.heapify(pq)  # ensure heap property
    exported = set(map(str, exp_data))
    enqueued = set(map(str, enq_data))
    return pq, exported, enqueued

def _scan_exported_from_disk(users_dir: str) -> Set[str]:
    """
    If no checkpoint exists, treat users as 'exported' if their reviews.json exists.
    This makes export idempotent and safe to re-run a partially completed user.
    """
    done = set()
    if not os.path.isdir(users_dir):
        return done
    for name in os.listdir(users_dir):
        udir = build_path(users_dir, name)
        if not os.path.isdir(udir):
            continue
        reviews_fp = build_path(udir, "reviews") + ".json"
        if os.path.exists(reviews_fp):
            done.add(name)
    return done

# --- Enqueue helpers -----------------------------------------------------

def _item_to_username_and_url(item) -> Tuple[Optional[str], Optional[str]]:
    if isinstance(item, str):
        u = item.strip().strip("/")
        return (u or None), (f"https://letterboxd.com/{u}/" if u else None)
    if isinstance(item, dict):
        for k in ("username", "name", "user", "slug"):
            if k in item and item[k]:
                u = str(item[k]).strip().strip("/")
                if u:
                    url = item.get("profile_url") or item.get("url") or f"https://letterboxd.com/{u}/"
                    return u, url
        for blk in ("user", "profile"):
            sub = item.get(blk)
            if isinstance(sub, dict):
                for k in ("username", "name", "slug"):
                    if k in sub and sub[k]:
                        u = str(sub[k]).strip().strip("/")
                        if u:
                            url = sub.get("url") or f"https://letterboxd.com/{u}/"
                            return u, url
        return None, None
    for attr in ("username", "name", "slug"):
        if hasattr(item, attr):
            u = str(getattr(item, attr) or "").strip().strip("/")
            if u:
                url = getattr(item, "profile_url", None) or getattr(item, "url", None) or f"https://letterboxd.com/{u}/"
                return u, url
    return None, None

def _priority_for_new_user(username: str) -> int:
    if not LOOKUP_FOLLOWER_COUNT_FOR_NEW:
        return 0
    try:
        u = lb_user.User(username)
        v = extract_followers_count(u)
        return int(v) if v is not None else 0
    except Exception:
        return 0

def enqueue_new_from_iterable(
    pq: List[PQItem],
    items: Iterable,
    enqueued: Set[str],
    exported: Set[str],
    max_new: int = MAX_NEW_PER_USER,
) -> int:
    added = 0
    for it in items or []:
        if added >= max_new:
            break
        u, url = _item_to_username_and_url(it)
        if not u:
            continue
        if u in exported or u in enqueued:
            continue
        prio = _priority_for_new_user(u)
        heapq.heappush(pq, (-prio, u, url or f"https://letterboxd.com/{u}/"))
        enqueued.add(u)
        added += 1
    return added

# --- export_user (idempotent) -------------------------------------------------

def export_user(username: str, users_dir: str):
    """Export following, followers count, and reviews for one user. Returns `following` raw list."""
    sleep_polite(1, 2)
    t0 = time.time()
    print(f"\n→ {username}: starting…")
    try:
        u = lb_user.User(username)
    except Exception as e:
        print(f"  ! failed to init user '{username}': {e!r}")
        return None

    user_dir = build_path(users_dir, u.username)
    check_and_create_dirs([user_dir])

    sleep_polite(1, 2)

    # Following
    following = None
    t = time.time()
    try:
        following = lb_user.User.get_following(u)
        path = build_path(user_dir, "following")
        save_json(path, following)
        print(f"  {time.time() - t:5.2f}s - following       → {build_click_url(path)}.json")
    except Exception as e:
        print(f"  ! get_following failed: {e!r}")

    sleep_polite(1, 2)

    # Followers COUNT only
    t = time.time()
    try:
        follower_count = extract_followers_count(u)
        path = build_path(user_dir, "followers_count")
        save_json(path, {"username": u.username, "followers_count": int(follower_count or 0)})
        print(
            f"  {time.time() - t:5.2f}s - followers_count  → {build_click_url(path)}.json "
            f"(count={follower_count})"
        )
    except Exception as e:
        print(f"  ! followers_count failed: {e!r}")

    sleep_polite(1, 2)

    # Reviews
    t = time.time()
    try:
        all_reviews = lb_user.User.get_reviews(u)
        path = build_path(user_dir, "reviews")
        save_json(path, all_reviews)
        print(
            f"  {time.time() - t:5.2f}s - reviews (all)    → {build_click_url(path)}.json "
            f"({len(all_reviews) if all_reviews is not None else 0} total)"
        )
    except Exception as e:
        print(f"  ! get_reviews failed: {e!r}")

    print(f"← {username}: done in {time.time() - t0:.2f}s")
    return following

# -------- Main with resume support --------
def main() -> None:
    start = time.time()
    fresh = ("--fresh" in sys.argv)

    # Prepare export + state dirs
    root = os.getcwd()
    exports = build_path(root, "exports")
    users_dir = build_path(exports, "users")
    state_dir = build_path(exports, "_state")
    check_and_create_dirs([exports, users_dir, state_dir])
    print("Export directories ready.")

    # Try to resume
    resumed = False
    loaded = None if fresh else _load_checkpoint(state_dir)

    if loaded:
        pq, exported, enqueued = loaded
        resumed = True
        print(f"Resuming from checkpoint: {len(exported)} exported, {len(pq)} queued.")
    else:
        # New run or no checkpoint: build from seed + disk scan
        exported = _scan_exported_from_disk(users_dir)
        pq = read_seed_pq("seed.csv")
        # Filter PQ against already exported
        if exported:
            pq = [item for item in pq if item[1] not in exported]
            heapq.heapify(pq)
        enqueued = set(u for _, u, _ in pq)
        if exported:
            print(f"Found {len(exported)} previously completed users on disk (via reviews.json).")
        if not pq:
            print("No usernames queued from seed.csv. If this is unexpected, check seed.csv.")
        # Save initial checkpoint so we can resume even if interrupted early
        _save_checkpoint(state_dir, pq, exported, enqueued)

    processed = len(exported)  # for display only

    try:
        while pq and processed - len(exported) < PROCESSED_LIMIT or pq:
            total_known = len(pq) + len(exported)
            pad = len(str(max(total_known, 1)))

            username, priority, profile_url = pq_pop(pq)
            if username in exported:
                continue

            processed += 1
            print(f"[{processed:0>{pad}}/{total_known}] {username} (priority={priority})")

            following_raw = export_user(username, users_dir)
            exported.add(username)

            # Immediately enqueue newly discovered nodes
            if following_raw:
                added = enqueue_new_from_iterable(
                    pq=pq,
                    items=following_raw,
                    enqueued=enqueued,
                    exported=exported,
                    max_new=MAX_NEW_PER_USER,
                )
                if added:
                    print(f"  ↳ queued {added} new profiles from {username}'s following")

            # Save checkpoint after each user
            _save_checkpoint(state_dir, pq, exported, enqueued)

    except KeyboardInterrupt:
        print("\nInterrupted by user. Saving checkpoint and exiting…")
        _save_checkpoint(state_dir, pq, exported, enqueued)
        sys.exit(130)

    # Finished normally
    _save_checkpoint(state_dir, pq, exported, enqueued)
    print("\nAll done!")
    print(f"  Total processed (completed on disk): {len(exported)}")
    print(f"  Still queued:                      : {len(pq)}")
    print(f"  Total time:                        : {time.time() - start:.2f}s")
    print(f"  Outputs at:                        : {build_click_url(users_dir)}")
    if resumed:
        print("  (This run resumed from a previous checkpoint.)")

if __name__ == "__main__":
    main()

## Scraping the Movies

In [ ]:
MAX_RETRIES = 3

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

# --------------------------------------------------------------------------- #
# Helpers
# --------------------------------------------------------------------------- #

def project_root() -> Path:
    return Path(__file__).resolve().parents[2]


def slugify_title(title: str) -> str:
    s = title.lower()
    s = s.replace("&", "and")
    s = s.replace("’", "").replace("'", "")
    s = re.sub(r"[^a-z0-9]+", "-", s)
    return re.sub(r"-{2,}", "-", s).strip("-")


def join_names(items, key="name"):
    return "|".join(str(it.get(key, "")).strip() for it in items if it.get(key)) if items else ""


def extract_movie_info(movie: Movie) -> dict:
    def attr(name, default=None):
        return getattr(movie, name, default)

    trailer = attr("trailer") or {}
    details = attr("details") or []
    genres = attr("genres") or []
    cast = attr("cast") or []
    crew = attr("crew") or {}
    popular_reviews = attr("popular_reviews") or []

    details_by_type = {}
    for d in details:
        t = d.get("type")
        if t:
            details_by_type.setdefault(t, []).append(d)

    directors = crew.get("director", []) or []
    writers = (
        crew.get("writer", [])
        or crew.get("screenplay", [])
        or crew.get("screenwriter", [])
        or []
    )
    producers = crew.get("producer", []) or []

    data = {
        "slug": attr("slug", ""),
        "url": attr("url", ""),
        "letterboxd_id": attr("letterboxd_id", ""),
        "title": attr("title", ""),
        "original_title": attr("original_title", ""),
        "year": attr("year", ""),
        "runtime": attr("runtime", ""),
        "tagline": attr("tagline", ""),
        "description": attr("description", ""),
        "poster": attr("poster", ""),
        "banner": attr("banner", ""),
        "tmdb_link": attr("tmdb_link", ""),
        "imdb_link": attr("imdb_link", ""),
        "rating": attr("rating", ""),
        "num_popular_reviews": len(popular_reviews),
        "genres": join_names(genres),
        "genre_slugs": "|".join(g.get("slug", "") for g in genres if g.get("slug")),
        "studios": join_names(details_by_type.get("studio", [])),
        "countries": join_names(details_by_type.get("country", [])),
        "languages": join_names(details_by_type.get("language", [])),
        "cast": join_names(cast),
        "directors": join_names(directors),
        "writers": join_names(writers),
        "producers": join_names(producers),
        "trailer_id": trailer.get("id", ""),
        "trailer_link": trailer.get("link", ""),
        "trailer_embed_url": trailer.get("embed_url", ""),
    }
    return data


# --------------------------------------------------------------------------- #
# Progress tracking
# --------------------------------------------------------------------------- #

def progress_file_for(output_csv: Path) -> Path:
    return output_csv.with_suffix(output_csv.suffix + ".progress")


def load_progress(progress_file: Path) -> int:
    if progress_file.exists():
        try:
            return int(progress_file.read_text().strip())
        except:
            return -1
    return -1


def save_progress(progress_file: Path, index: int):
    try:
        progress_file.write_text(str(index))
    except Exception as e:
        logger.warning(f"Failed to write progress file: {e}")


# --------------------------------------------------------------------------- #
# Scraper
# --------------------------------------------------------------------------- #

def scrape_movies(input_csv: Path, output_csv: Path, delay: float):
    # Load input
    with input_csv.open("r", encoding="utf-8", newline="") as f:
        rows = list(csv.DictReader(f))

    logger.info(f"Loaded {len(rows)} movies from {input_csv}")

    # Progress
    pfile = progress_file_for(output_csv)
    last_done = load_progress(pfile)
    start = last_done + 1

    if start >= len(rows):
        logger.info("All movies already scraped according to progress.")
        return

    logger.info(f"Resuming at index {start} (human index {start+1})")

    # Fields
    fieldnames = [
        "movie_name",
        "number_of_reviews",
        "slug",
        "url",
        "letterboxd_id",
        "title",
        "original_title",
        "year",
        "runtime",
        "tagline",
        "description",
        "poster",
        "banner",
        "tmdb_link",
        "imdb_link",
        "rating",
        "num_popular_reviews",
        "genres",
        "genre_slugs",
        "studios",
        "countries",
        "languages",
        "cast",
        "directors",
        "writers",
        "producers",
        "trailer_id",
        "trailer_link",
        "trailer_embed_url",
    ]

    write_header = not output_csv.exists() or last_done < 0

    # Open output for append or overwrite
    with output_csv.open("a" if output_csv.exists() else "w", encoding="utf-8", newline="") as fout:
        writer = csv.DictWriter(fout, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
            fout.flush()

        for i in range(start, len(rows)):
            human = i + 1
            movie_name = rows[i].get("movie_name") or rows[i].get("moviename") or ""

            logger.info(f"({human:04d}) Fetching '{movie_name}'")

            if not movie_name:
                logger.warning(f"({human:04d}) empty movie name → skipping")
                save_progress(pfile, i)
                continue

            slug = slugify_title(movie_name)

            # Try scraping with retries. If it doesn't work: SKIP.
            movie_data = None
            for attempt in range(1, MAX_RETRIES + 1):
                try:
                    m = Movie(slug)
                    movie_data = extract_movie_info(m)
                    break
                except Exception as e:
                    logger.warning(
                        f"({human:04d}) attempt {attempt}/{MAX_RETRIES} failed "
                        f"for '{movie_name}': {e}"
                    )
                    time.sleep(delay)

            if movie_data is None:
                logger.error(f"({human:04d}) FAILED permanently → skipping movie")
                save_progress(pfile, i)
                continue

            # Write result
            out_row = {
                "movie_name": movie_name,
                "number_of_reviews": rows[i].get("number_of_reviews", ""),
                **movie_data,
            }
            writer.writerow(out_row)
            fout.flush()

            # Update progress
            save_progress(pfile, i)

            time.sleep(delay)


# --------------------------------------------------------------------------- #
# CLI
# --------------------------------------------------------------------------- #

def main():
    proj = project_root()

    default_in = proj / "Data" / "CSV" / "reviews_per_movie.csv"
    default_out = proj / "Data" / "CSV" / "movies.csv"

    parser = argparse.ArgumentParser()
    parser.add_argument("--input", type=Path, default=default_in)
    parser.add_argument("--output", type=Path, default=default_out)
    parser.add_argument("--delay", type=float, default=1.0)

    args = parser.parse_args()

    print("=== Letterboxd Movie Scraper ===")
    print(f"Input:  {args.input}")
    print(f"Output: {args.output}")
    print(f"Delay:  {args.delay}s\n")

    try:
        scrape_movies(args.input, args.output, args.delay)
    except KeyboardInterrupt:
        print("\nInterrupted, exiting.")


if __name__ == "__main__":
    main()

# Part 1: Bulding the Network

In [3]:
# -------------------------------------------------------------------------
# Logging
# -------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger("letterboxd_network")

# -------------------------------------------------------------------------
# Letterboxd Data Parser
# -------------------------------------------------------------------------
class LetterboxdDataParser:
    """Parser for Letterboxd exported data."""

    def __init__(self, exports_path: str = "exports"):
        """
        Initialize the parser.

        Args:
            exports_path: Path to the exports directory
        """
        self.exports_path = Path(exports_path)
        self.users_path = self.exports_path / "users"
        self.state_path = self.exports_path / "_state"

        if not self.exports_path.exists():
            raise FileNotFoundError(f"Exports path not found: {self.exports_path}")

    def get_exported_users(self) -> List[str]:
        """
        Get list of exported usernames from exported.json.

        Returns:
            List of usernames that have been successfully exported
        """
        exported_file = self.state_path / "exported.json"

        if not exported_file.exists():
            logger.warning(f"exported.json not found at {exported_file}")
            return []

        try:
            with open(exported_file, "r", encoding="utf-8") as f:
                users = json.load(f)
            logger.info(f"Found {len(users)} exported users")
            return users
        except json.JSONDecodeError as e:
            logger.error(f"Error parsing exported.json: {e}")
            return []

    def load_user_following(self, username: str) -> Dict:
        """
        Load following.json for a specific user.

        Args:
            username: The username to load following data for

        Returns:
            Dictionary with following data, empty dict if not found
        """
        following_file = self.users_path / username / "following.json"

        if not following_file.exists():
            logger.warning(f"following.json not found for user {username}")
            return {}

        try:
            with open(following_file, "r", encoding="utf-8") as f:
                return json.load(f)
        except json.JSONDecodeError as e:
            logger.error(f"Error parsing following.json for {username}: {e}")
            return {}

    def load_user_metadata(self, username: str) -> Dict:
        """
        Load followers_count.json for a specific user.

        Args:
            username: The username to load metadata for

        Returns:
            Dictionary with metadata (username, followers_count)
        """
        metadata_file = self.users_path / username / "followers_count.json"

        if not metadata_file.exists():
            logger.warning(f"followers_count.json not found for user {username}")
            return {"username": username, "followers_count": 0}

        try:
            with open(metadata_file, "r", encoding="utf-8") as f:
                return json.load(f)
        except json.JSONDecodeError as e:
            logger.error(f"Error parsing followers_count.json for {username}: {e}")
            return {"username": username, "followers_count": 0}

    def load_user_reviews(self, username: str) -> Dict:
        """
        Load reviews.json for a specific user.

        Args:
            username: The username to load reviews for

        Returns:
            Dictionary with reviews data, empty dict if not found
        """
        reviews_file = self.users_path / username / "reviews.json"

        if not reviews_file.exists():
            logger.warning(f"reviews.json not found for user {username}")
            return {}

        try:
            with open(reviews_file, "r", encoding="utf-8") as f:
                data = json.load(f)
                # Return the reviews dict, handle both formats
                if isinstance(data, dict) and "reviews" in data:
                    return data["reviews"]
                return data
        except json.JSONDecodeError as e:
            logger.error(f"Error parsing reviews.json for {username}: {e}")
            return {}

    def get_all_user_data(self) -> Dict:
        """
        Load all data for all exported users.

        Returns:
            Dictionary with structure:
            {
                "username": {
                    "followers_count": int,
                    "following": dict,
                    "reviews": dict
                }
            }
        """
        users = self.get_exported_users()
        all_data = {}

        logger.info(f"Loading data for {len(users)} users...")

        for i, username in enumerate(users, 1):
            if i % 50 == 0:
                logger.info(f"Progress: {i}/{len(users)} users loaded")

            metadata = self.load_user_metadata(username)
            following = self.load_user_following(username)
            reviews = self.load_user_reviews(username)

            all_data[username] = {
                "followers_count": metadata.get("followers_count", 0),
                "following": following,
                "reviews": reviews,
            }

        logger.info(f"Successfully loaded data for {len(all_data)} users")
        return all_data


# -------------------------------------------------------------------------
# Network Builder
# -------------------------------------------------------------------------
def build_letterboxd_network(parser: LetterboxdDataParser) -> nx.DiGraph:
    """
    Build a NetworkX directed graph from Letterboxd data.

    The graph has the following properties:
    - Nodes: Users (only exported users with complete data)
    - Edges: Directed follows (A -> B means "A follows B")
    - Node attributes: username, followers_count, reviews

    Args:
        parser: LetterboxdDataParser instance

    Returns:
        NetworkX DiGraph with user nodes and follow edges
    """
    logger.info("Building Letterboxd network...")

    # Load all user data
    all_data = parser.get_all_user_data()
    exported_users = set(all_data.keys())

    logger.info(f"Creating graph with {len(exported_users)} users")

    # Create directed graph
    G = nx.DiGraph()

    # Add nodes with attributes
    logger.info("Adding nodes with attributes...")
    for username, data in all_data.items():
        G.add_node(
            username,
            username=username,
            followers_count=data["followers_count"],
            reviews=data["reviews"],
        )

    logger.info(f"Added {G.number_of_nodes()} nodes")

    # Add edges (follows)
    logger.info("Adding edges (follows)...")
    edge_count = 0
    skipped_edges = 0

    for username, data in all_data.items():
        following = data["following"]

        for followed_username in following.keys():
            # Only add edge if the followed user is also in our exported users
            if followed_username in exported_users:
                G.add_edge(username, followed_username)
                edge_count += 1
            else:
                skipped_edges += 1

    logger.info(f"Added {edge_count} edges")
    logger.info(f"Skipped {skipped_edges} edges to non-exported users")

    # Log graph statistics
    logger.info("=" * 50)
    logger.info("Network Statistics:")
    logger.info(f"  Nodes: {G.number_of_nodes()}")
    logger.info(f"  Edges: {G.number_of_edges()}")
    logger.info(
        f"  Average degree: {sum(dict(G.degree()).values()) / G.number_of_nodes():.2f}"
    )
    logger.info(
        f"  Average in-degree: {sum(dict(G.in_degree()).values()) / G.number_of_nodes():.2f}"
    )
    logger.info(
        f"  Average out-degree: {sum(dict(G.out_degree()).values()) / G.number_of_nodes():.2f}"
    )
    logger.info("=" * 50)

    return G


# -------------------------------------------------------------------------
# Save / Load Helpers
# -------------------------------------------------------------------------
def save_network(graph: nx.DiGraph, output_path: str = "letterboxd_network.pickle") -> None:
    """
    Save the network graph to a pickle file.

    Args:
        graph: NetworkX DiGraph to save
        output_path: Path where to save the pickle file
    """
    output_file = Path(output_path)

    logger.info(f"Saving network to {output_file}...")

    try:
        with open(output_file, "wb") as f:
            pickle.dump(graph, f)

        file_size = output_file.stat().st_size / (1024 * 1024)  # MB
        logger.info(f"Network saved successfully! File size: {file_size:.2f} MB")
    except Exception as e:
        logger.error(f"Error saving network: {e}")
        raise


def load_network(input_path: str = "letterboxd_network.pickle") -> nx.DiGraph:
    """
    Load a network graph from a pickle file.

    Args:
        input_path: Path to the pickle file

    Returns:
        NetworkX DiGraph loaded from file
    """
    input_file = Path(input_path)

    if not input_file.exists():
        raise FileNotFoundError(f"Network file not found: {input_file}")

    logger.info(f"Loading network from {input_file}...")

    try:
        with open(input_file, "rb") as f:
            graph = pickle.load(f)

        logger.info("Network loaded successfully!")
        logger.info(f"  Nodes: {graph.number_of_nodes()}")
        logger.info(f"  Edges: {graph.number_of_edges()}")

        return graph
    except Exception as e:
        logger.error(f"Error loading network: {e}")
        raise


def build_and_save_network(
    exports_path: str = "./State_1",
    output_path: str = "./letterboxd_network.pickle",
) -> nx.DiGraph:
    """
    Convenience helper for notebooks:
    - Initializes parser
    - Builds network
    - Saves pickle
    - Prints basic stats

    Returns:
        The built NetworkX DiGraph
    """
    logger.info("Initializing parser...")
    parser = LetterboxdDataParser(exports_path)

    logger.info("Building network...")
    graph = build_letterboxd_network(parser)

    print("\n" + "=" * 60)
    print("LETTERBOXD NETWORK BUILT SUCCESSFULLY")
    print("=" * 60)
    print(f"Total Nodes (Users): {graph.number_of_nodes()}")
    print(f"Total Edges (Follows): {graph.number_of_edges()}")
    print(
        f"Average Degree: {sum(dict(graph.degree()).values()) / graph.number_of_nodes():.2f}"
    )
    print("=" * 60 + "\n")

    save_network(graph, output_path)
    print(f"✓ Network saved to {output_path}")

    print("\n" + "=" * 60)
    print("HOW TO USE THE NETWORK")
    print("=" * 60)
    print("import pickle")
    print("import networkx as nx")
    print("")
    print("# Load the network")
    print(f"with open('{output_path}', 'rb') as f:")
    print("    G = pickle.load(f)")
    print("")
    print("# Get node data")
    print("node_data = G.nodes['aadowd']")
    print("print(f\"Followers: {node_data['followers_count']}\")")
    print("print(f\"Reviews: {len(node_data['reviews'])}\")")
    print("")
    print("# Get degree")
    print("print(f\"Following: {G.out_degree('aadowd')}\")")
    print("print(f\"Followed by (in network): {G.in_degree('aadowd')}\")")
    print("=" * 60 + "\n")

    return graph

G = build_and_save_network()

2025-12-03 16:26:05,401 - letterboxd_network - INFO - Initializing parser...
2025-12-03 16:26:05,403 - letterboxd_network - INFO - Building network...
2025-12-03 16:26:05,403 - letterboxd_network - INFO - Building Letterboxd network...
2025-12-03 16:26:05,404 - letterboxd_network - INFO - Found 904 exported users
2025-12-03 16:26:05,404 - letterboxd_network - INFO - Loading data for 904 users...
2025-12-03 16:26:05,467 - letterboxd_network - WARNING - reviews.json not found for user a24
2025-12-03 16:26:05,491 - letterboxd_network - WARNING - reviews.json not found for user aaron_leckey001
2025-12-03 16:26:05,549 - letterboxd_network - WARNING - reviews.json not found for user afi
2025-12-03 16:26:05,550 - letterboxd_network - WARNING - followers_count.json not found for user afragabrie1001
2025-12-03 16:26:05,550 - letterboxd_network - WARNING - following.json not found for user afragabrie1001
2025-12-03 16:26:05,550 - letterboxd_network - WARNING - reviews.json not found for user afr


LETTERBOXD NETWORK BUILT SUCCESSFULLY
Total Nodes (Users): 904
Total Edges (Follows): 34743
Average Degree: 76.87



2025-12-03 16:26:16,029 - letterboxd_network - INFO - Network saved successfully! File size: 375.68 MB


✓ Network saved to ./letterboxd_network.pickle

HOW TO USE THE NETWORK
import pickle
import networkx as nx

# Load the network
with open('./letterboxd_network.pickle', 'rb') as f:
    G = pickle.load(f)

# Get node data
node_data = G.nodes['aadowd']
print(f"Followers: {node_data['followers_count']}")
print(f"Reviews: {len(node_data['reviews'])}")

# Get degree
print(f"Following: {G.out_degree('aadowd')}")
print(f"Followed by (in network): {G.in_degree('aadowd')}")

